In [2]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from sklearn.utils import resample
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')
mean_spf_trim_nr = pd.read_csv('cleaned_data/mean_spf_trim_nr.csv')
ind_spf_trim_nr = pd.read_csv('cleaned_data/ind_spf_trim_nr.csv')

mean_spf_trim['DATE'] = pd.to_datetime(mean_spf_trim['DATE'])
mean_spf_trim_nr['DATE'] = pd.to_datetime(mean_spf_trim_nr['DATE'])
ind_spf_trim['DATE'] = pd.to_datetime(ind_spf_trim['DATE'])
ind_spf_trim_nr['DATE'] = pd.to_datetime(ind_spf_trim_nr['DATE'])

ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')
ind_spf_trim_nr = ind_spf_trim_nr.dropna(subset='r_t1')

###############################################
                ## ESTIMATION ##
###############################################

### OLS ###
def OLS(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        revisions = df[f'r_t{j}']
        revisions = sm.add_constant(revisions)
        errors = df[f'e_t{j}']
        initial = sm.OLS(errors, revisions).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=1))
    return regs


### ID FE ###
def ID_FE(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=1))
    return regs


### Two-way Fixed Effects ###
def IDT_FE(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float), pd.get_dummies(df['DATE'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=1))
    return regs


### AR Estimates ###
def AR(df, end_date):
    df = df.loc[(df['DATE'] >= '1965-06-30') & (df['DATE'] <= end_date)]  # Filter data
    grwth_t = df['t3']
    reg = AutoReg(grwth_t, 1).fit()
    return reg


### Parameter Estimates ###
def params(ols, pldols, fe, fe2, ar1):
    params = []
    params.append(ols / (1 + ols))
    params.append(1 / (1 + ols))
    params.append((-((2 * pldols) + 1) + np.sqrt(((2 * pldols) + 1)**2 - 4 * (pldols + (pldols * ar1**2) + 1) * pldols)) / (2 * (pldols + (pldols * ar1**2) + 1)))
    params.append((-((2 * fe) + 1) + np.sqrt(((2 * fe) + 1)**2 - 4 * (fe + (fe * ar1**2) + 1) * fe)) / (2 * (fe + (fe * ar1**2) + 1)))
    params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))
    return params


def compute_regs(date, mean, ind):
    mean_regs = OLS(mean, f'{date}')
    ind_regs = OLS(ind, f'{date}')
    ind_regs_fe = ID_FE(ind, f'{date}')
    ind_regs_fe2 = IDT_FE(ind, f'{date}')
    ar_1 = AR(vintage_trim, f'{date}')
    regs = [mean_regs, ind_regs, ind_regs_fe, ind_regs_fe2, ar_1]
    parameters = params(mean_regs[3].params[1], ind_regs[3].params[1], ind_regs_fe[3].params[1], ind_regs_fe2[3].params[1], ar_1.roots)
    return regs, parameters


def new_2022(df):
    a = df[(df['DATE'] >= '2021-09-30') & (df['DATE'] <= '2022-09-30')]
    return df[~df.index.isin(a.index)]

mean_spf_trim22 = new_2022(mean_spf_trim)
ind_spf_trim22 = new_2022(ind_spf_trim)

regs_2016, parameters_2016 = compute_regs('2016-12-31', mean_spf_trim, ind_spf_trim)
regs_2016_nr, parameters_2016_nr = compute_regs('2016-12-31', mean_spf_trim_nr, ind_spf_trim_nr)
%store regs_2016
%store parameters_2016
%store regs_2016_nr
%store parameters_2016_nr

regs_2019, parameters_2019 = compute_regs('2019-12-31', mean_spf_trim, ind_spf_trim)
regs_2019_nr, parameters_2019_nr = compute_regs('2019-12-31', mean_spf_trim_nr, ind_spf_trim_nr)
%store regs_2019
%store parameters_2019
%store regs_2019_nr
%store parameters_2019_nr

regs_2022, parameters_2022 = compute_regs('2022-12-31', mean_spf_trim, ind_spf_trim)
regs_2022_nr, parameters_2022_nr = compute_regs('2022-12-31', mean_spf_trim_nr, ind_spf_trim_nr)
%store regs_2022
%store parameters_2022
%store regs_2022_nr
%store parameters_2022_nr

regs_2022n, parameters_2022n = compute_regs('2022-12-31', mean_spf_trim22, ind_spf_trim22)
%store regs_2022n
%store parameters_2022n

mean_spf_trim22.to_csv('cleaned_data/mean_spf_trim22.csv', sep=',', index=True)
ind_spf_trim22.to_csv('cleaned_data/ind_spf_trim22.csv', sep=',', index=True)

/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/var/folders/6q/zww1c4xj04d9c7jlxyz1zc6c0000gn/T/ipykernel_46012/1887521801.py:84: RuntimeWarning: invalid value encountered in sqrt
  params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))
/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/var/folders/6q/zww1c4xj04d9c7jlxyz1zc6c0000gn/T/ipykernel_46012/1887521801.py:84: RuntimeWarning: invalid value encountered in sqrt
  params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))


Stored 'regs_2016' (list)
Stored 'parameters_2016' (list)
Stored 'regs_2016_nr' (list)
Stored 'parameters_2016_nr' (list)


/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/var/folders/6q/zww1c4xj04d9c7jlxyz1zc6c0000gn/T/ipykernel_46012/1887521801.py:84: RuntimeWarning: invalid value encountered in sqrt
  params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))
/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/var/folders/6q/zww1c4xj04d9c7jlxyz1zc6c0000gn/T/ipykernel_46012/1887521801.py:84: RuntimeWarning: invalid value encountered in sqrt
  params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))


Stored 'regs_2019' (list)
Stored 'parameters_2019' (list)
Stored 'regs_2019_nr' (list)
Stored 'parameters_2019_nr' (list)


/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/var/folders/6q/zww1c4xj04d9c7jlxyz1zc6c0000gn/T/ipykernel_46012/1887521801.py:84: RuntimeWarning: invalid value encountered in sqrt
  params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))
/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Stored 'regs_2022' (list)
Stored 'parameters_2022' (list)
Stored 'regs_2022_nr' (list)
Stored 'parameters_2022_nr' (list)


/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/var/folders/6q/zww1c4xj04d9c7jlxyz1zc6c0000gn/T/ipykernel_46012/1887521801.py:84: RuntimeWarning: invalid value encountered in sqrt
  params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))


Stored 'regs_2022n' (list)
Stored 'parameters_2022n' (list)


In [16]:
regs_2022_nr[3][3].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.900
Model:                            OLS   Adj. R-squared:                  0.891
Method:                 Least Squares   F-statistic:                     337.0
Date:                Fri, 26 Apr 2024   Prob (F-statistic):               0.00
Time:                        16:18:16   Log-Likelihood:                 14285.
No. Observations:                3492   AIC:                        -2.801e+04
Df Residuals:                    3214   BIC:                        -2.630e+04
Df Model:                         277                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0229      0.003     -8.879      0.000      -0.028      -0.018
x1            -0.4799      0.026    -18.576      0.000      -0.531      -0.429
x2            -0.0018      0.002     -0.754      0.451      -0.006       0.003
x3             0.0009      0.002      0.405      0.686      -0.003       0.005
x4          -8.21e-05      0.001     -0.056      0.955      -0.003       0.003
x5             0.0019      0.001      1.433      0.152      -0.001       0.004
x6             0.0012      0.002      0.681      0.496      -0.002       0.005
x7            -0.0045      0.002     -2.320      0.020      -0.008      -0.001
x8             0.0028      0.003      1.041      0.298      -0.002       0.008
x9             0.0005      0.005      0.101      0.919      -0.010       0.011
x10           -0.0009      0.002     -0.522      0.601      -0.004       0.002
x11           -0.0004      0.001     -0.290      0.772      -0.003       0.002
x12            0.0099      0.005      1.798      0.072      -0.001       0.021
x13            0.0024      0.001      1.776      0.076      -0.000       0.005
x14            0.0024      0.002      1.497      0.135      -0.001       0.005
x15            0.0037      0.005      0.811      0.417      -0.005       0.013
x16            0.0036      0.002      1.469      0.142      -0.001       0.008
x17            0.0003      0.001      0.216      0.829      -0.002       0.003
x18            0.0012      0.002      0.732      0.464      -0.002       0.005
x19            0.0057      0.002      3.648      0.000       0.003       0.009
x20           -0.0014      0.002     -0.782      0.434      -0.005       0.002
x21            0.0038      0.003      1.497      0.135      -0.001       0.009
x22            0.0002      0.001      0.125      0.900      -0.002       0.003
x23           -0.0009      0.002     -0.537      0.592      -0.004       0.002
x24            0.0024      0.002      1.543      0.123      -0.001       0.005
x25            0.0062      0.002      3.635      0.000       0.003       0.010
x26           -0.0013      0.002     -0.540      0.589      -0.006       0.003
x27            0.0015      0.001      1.053      0.293      -0.001       0.004
x28            0.0024      0.002      1.288      0.198      -0.001       0.006
x29           -0.0055      0.001     -3.704      0.000      -0.008      -0.003
x30        -1.088e-05      0.001     -0.008      0.994      -0.003       0.003
x31           -0.0039      0.003     -1.185      0.236      -0.010       0.003
x32            0.0028      0.001      2.046      0.041       0.000       0.005
x33            0.0048      0.002      2.256      0.024       0.001       0.009
x34           -0.0028      0.002     -1.161      0.246      -0.007       0.002
x35            0.0027      0.001      1.898      0.058   -8.91e-05       0.006
x3

In [6]:
###############################################
              ## ID BOOTSTRAP ##
###############################################
def boot(df, R, date):
    RR = R + 1
    boot_regs0 = []
    boot_regs1 = []
    boot_regs2 = []
    boot_regs3 = []
    for i in range(1, RR):
        boot = resample(df['ID'], replace=True, n_samples=len(df), random_state=i)
        boot_df = df[df['ID'].isin(boot)]
        boot_ols = OLS(boot_df, date)
        boot_regs0.append(boot_ols[0].params[1])
        boot_regs1.append(boot_ols[1].params[1])
        boot_regs2.append(boot_ols[2].params[1])
        boot_regs3.append(boot_ols[3].params[1])
    boot_regs = [boot_regs0, boot_regs1, boot_regs2, boot_regs3]
    return boot_regs

boot_regs_2016 = boot(ind_spf_trim, 10000, '2016-12-31')
%store boot_regs_2016

boot_regs_2019 = boot(ind_spf_trim, 10000, '2019-12-31')
%store boot_regs_2019

boot_regs_2022 = boot(ind_spf_trim, 10000, '2022-12-31')
%store boot_regs_2022

boot_regs_2022n = boot(ind_spf_trim22, 10000, '2022-12-31')
%store boot_regs_2022n

Stored 'boot_regs_2016' (list)
Stored 'boot_regs_2019' (list)
Stored 'boot_regs_2022' (list)
Stored 'boot_regs_2022n' (list)


In [7]:
%store -r boot_regs_2016
%store -r boot_regs_2019
%store -r boot_regs_2022
%store -r boot_regs_2022n
print(np.quantile(boot_regs_2016[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2016[3], 0.5))
print(np.quantile(boot_regs_2019[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2019[3], 0.5))
print(np.quantile(boot_regs_2022[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2022[3], 0.5))
print(np.quantile(boot_regs_2022n[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2022n[3], 0.5))

[-0.19400186 -0.19146638]
-0.19176521394011467
[-0.20467117 -0.20218268]
-0.20245905135330092
[0.0602634 0.0625437]
0.062246529362919856
[-0.13942712 -0.13702902]
-0.1373404364100622


In [4]:
###############################################
    ## FORECASTER-BY-FORECASTER BOOTSTRAP ##
###############################################
